# Instruction fine-tuning on two GPUs

Configure the run, inspect the dataset and environment, and launch the standalone Accelerate training script. Training runs in separate processes; the model is saved to the output directory.


## 1. Setup and configuration
Run from the repository or a subdirectory of it. Change the settings below before launching.


In [1]:
from pathlib import Path
import importlib.util
import json
import os
import subprocess
import sys

project_root = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / "model_training").is_dir()),
    None,
)

if project_root is None:
    raise RuntimeError("Run this notebook from inside the deep_learning repository")

In [2]:

file_name = project_root / "datasets/instruction_tuning/instruction-data.json"
output_dir = project_root / "execution_results/qwen_instruction"
data_dir = project_root / "datasets"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)


In [3]:
# script = project_root / "model_training/post_training/02-sft/train_instruction.py"

In [4]:
import torch
from torch.utils.data import DataLoader, Subset
# from model_training.framework.data.post_train.dataloader import CyclingDataLoader
from datetime import datetime
from model_training.framework.train.tracking.logger import ExperimentLogger


In [5]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"


In [6]:
## Accelerator for Pytorch
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
device

'cuda'

## 2. Check environment and data
The JSON file must already exist. The original instruction-tuning notebook downloads it. The script uses the first 85% for training.


In [7]:
! nvidia-smi

overwriting variable ['FWDLLM_USER', 'GOTOOLCHAIN']
/nethome/dgarg39/.bashrc: line 43: /nethome/dgarg39/.cargo/env: No such file or directory
Wed Nov 18 00:57:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A40                     On  |   00000000:01:00.0 Off |                    0 |
|  0%   26C    P8             30W /  300W |       4MiB /  46068MiB |      0%      Default 

In [8]:
print("[STAGE] environment and dataset")
missing = [name for name in ("torch", "transformers", "accelerate") if importlib.util.find_spec(name) is None]
if missing:
    raise ModuleNotFoundError(f"Missing packages in the kernel environment: {', '.join(missing)}")
# if not script.is_file():
    # raise FileNotFoundError(f"Training script not found: {script}")
if not file_name.is_file():
    raise FileNotFoundError(f"Dataset not found: {file_name}; download it using the instruction-tuning notebook")
with file_name.open(encoding="utf-8") as f:
    data = json.load(f)
if not isinstance(data, list) or not data:
    raise ValueError("Dataset must be a non-empty JSON list")
import torch
print(f"✓ examples: {len(data)} | training split: {int(len(data) * 0.85)}")
print(f"✓ visible CUDA GPUs: {torch.cuda.device_count()}")

[STAGE] environment and dataset
✓ examples: 1100 | training split: 935
✓ visible CUDA GPUs: 8


In [9]:
from accelerate import Accelerator, notebook_launcher

from model_training.framework.data.post_train.dataloader import CyclingDataLoader
from model_training.framework.train.core.config import TrainConfig
from model_training.framework.train.core.train import TrainComponents, train_model
from model_training.framework.train.loss.sft import sft_forward, sft_loss
from model_training.framework.train.utils.lr_scheduler import cosine_lr
from model_training.framework.train.utils.optimizer import configure_optimizer

In [10]:
# num_gpus = 1
# batch_size = 8
# if torch.cuda.device_count() <= num_gpus:
#     raise RuntimeError(f"Requested {num_gpus} GPUs but fewer are visible to this kernel")
# print(f"✓ nominal effective batch size: {batch_size * num_gpus * accumulation_steps} examples")


## 3. Load the dataset

In [11]:
from pathlib import Path
import sys

sft_dir = project_root / "model_training" / "post_training" / "03-sft"
modules_file = sft_dir / "src" / "modules.py"

if not modules_file.is_file():
    raise FileNotFoundError(f"Missing source module: {modules_file}")

sys.path.insert(0, str(sft_dir))

from src.modules import return_dataset

In [12]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

pad_token_id = tokenizer.pad_token_id
pad_token_id

151643

In [13]:
train_portion = int(len(data) * 0.85)  # 85% for training
test_portion = int(len(data) * 0.1)    # 10% for testing
val_portion = len(data) - train_portion - test_portion  # Remaining 5% for validation
num_workers = 0
batch_size = 8

torch.manual_seed(123)


## 4. Run training
This cell starts the actual training job and waits for it to finish. A failed worker raises an error in the notebook.

In [14]:
text_to_test = "Every effort moves you"
torch.manual_seed(123)
SFT_CONFIG = {
    "num_processes": 2,          # set 1, 2, 4, ... GPUs here
    "mixed_precision": "fp16",  # "no", "fp16", or "bf16"
    "per_device_batch_size": 1,
    "grad_accum_steps": 4,
    "seq_length": 1024,
    "num_epochs": 2,
    "learning_rate": 1e-5,
    "weight_decay": 0.1,
    "warmup_steps": 0,
    "debug_examples": 80,
}

In [15]:
def train(text: str, cfg: dict) -> None:
    accelerator = Accelerator(mixed_precision=cfg["mixed_precision"])

    train_loader, val_loader, test_loader = return_dataset(
        MODEL_NAME,
        data_dir,
        file_name,
        url,
        tokenizer,
        train_portion,
        test_portion,
        pad_token_id,
        num_workers,
        cfg["per_device_batch_size"],
        accelerator.device,
    )

    base_loader = train_loader.loader

    small_loader = DataLoader(
        Subset(base_loader.dataset, range(cfg["debug_examples"])),
        batch_size=base_loader.batch_size,
        collate_fn=base_loader.collate_fn,
        shuffle=True,
        drop_last=True,
        num_workers=base_loader.num_workers,
    )

    train_loader_subset = CyclingDataLoader(
        small_loader,
        seq_len=cfg["seq_length"],
    )

    local_batches_per_epoch = math.ceil(
        len(train_loader_subset.loader) / accelerator.num_processes
    )
    steps_per_epoch = local_batches_per_epoch // cfg["grad_accum_steps"]

    if steps_per_epoch < 1:
        raise ValueError(
            "Not enough debug examples for the selected GPU count, "
            "batch size, and accumulation steps."
        )

    train_config = TrainConfig(
        num_steps=cfg["num_epochs"] * steps_per_epoch,
        learning_rate=cfg["learning_rate"],
        total_batch_size=(
            cfg["per_device_batch_size"]
            * accelerator.num_processes
            * cfg["seq_length"]
            * cfg["grad_accum_steps"]
        ),
        micro_batch_size=cfg["per_device_batch_size"],
        seq_length=cfg["seq_length"],
        accumulation_steps=cfg["grad_accum_steps"],
        max_lr=cfg["learning_rate"],
        min_lr=cfg["learning_rate"],
        warmup_steps=cfg["warmup_steps"],
        weight_decay=cfg["weight_decay"],
        device=accelerator.device,
        use_compile=False,
        precision=cfg["mixed_precision"],
    )

    model = return_model(MODEL_NAME)

    optimizer = configure_optimizer(
        model,
        train_config.weight_decay,
        train_config.learning_rate,
        master_process=accelerator.is_main_process,
        device=accelerator.device.type,
    )

    train_comp = TrainComponents(
        optimizer=optimizer,
        dataloader=train_loader_subset,
        scheduler=cosine_lr,
        forward_step=sft_forward,
        loss_fn=sft_loss,
    )

    logger = None
    if accelerator.is_main_process:
        run_name = datetime.now().strftime("%Y%m%d_%H%M%S")
        logger = ExperimentLogger(
            project="sft-experiments",
            log_dir=str(project_root / "execution_results" / "runs" / run_name),
        )

    train_model(
        model,
        train_components=train_comp,
        train_config=train_config,
        logger=logger,
        accelerator=accelerator,
    )

In [16]:
notebook_launcher(
    train,
    args=(text_to_test, SFT_CONFIG),
    num_processes=SFT_CONFIG["num_processes"],
    mixed_precision=SFT_CONFIG["mixed_precision"],
)

Launching training on 2 CUDAs.


W1118 00:58:13.409000 2992833 torch/multiprocessing/spawn.py:165] Terminating process 2993105 via signal SIGTERM
E1118 00:58:13.421000 2992833 torch/distributed/elastic/multiprocessing/api.py:836] failed (exitcode: 1) local_rank: 0 (pid: 2993104) of fn: train (start_method: fork)
E1118 00:58:13.421000 2992833 torch/distributed/elastic/multiprocessing/api.py:836] Traceback (most recent call last):
E1118 00:58:13.421000 2992833 torch/distributed/elastic/multiprocessing/api.py:836]   File "/coc/scratch/vchopra/learning_env/learning_env/lib/python3.13/site-packages/torch/distributed/elastic/multiprocessing/api.py", line 791, in _poll
E1118 00:58:13.421000 2992833 torch/distributed/elastic/multiprocessing/api.py:836]     self._pc.join(-1)
E1118 00:58:13.421000 2992833 torch/distributed/elastic/multiprocessing/api.py:836]     ~~~~~~~~~~~~~^^^^
E1118 00:58:13.421000 2992833 torch/distributed/elastic/multiprocessing/api.py:836]   File "/coc/scratch/vchopra/learning_env/learning_env/lib/python3

ChildFailedError: 
============================================================
train FAILED
------------------------------------------------------------
Failures:
  <NO_OTHER_FAILURES>
------------------------------------------------------------
Root Cause (first observed failure):
[0]:
  time      : 2026-11-18_00:58:13
  host      : inara.cc.gatech.edu
  rank      : 0 (local_rank: 0)
  exitcode  : 1 (pid: 2993104) 
  error_file: /tmp/torchelastic_2gr8t64n/none_7jhviwqw/attempt_0/0/error.json
  traceback : Traceback (most recent call last):
    File "/coc/scratch/vchopra/learning_env/learning_env/lib/python3.13/site-packages/torch/distributed/elastic/multiprocessing/errors/__init__.py", line 380, in wrapper
      return f(*args, **kwargs)
    File "/tmp/ipykernel_2992833/1684616313.py", line 2, in train
      accelerator = Accelerator(mixed_precision=cfg["mixed_precision"])
    File "/coc/scratch/vchopra/learning_env/learning_env/lib/python3.13/site-packages/accelerate/accelerator.py"

## 5. Inspect saved artifacts
The script saves the model and tokenizer on successful completion.


In [ ]:
# text_1 = "What is your name ? "
text_1 = "Every effort moves you ? What is this"
in_tokens = text_to_token_ids(text_1, tokenizer=tokenizer, 
                            #   allowed_special = {"<|endoftext|>"}
)
in_tokens=in_tokens.to(device)
print(in_tokens.shape)  # [1, sequence_length]

In [ ]:


out_tokens = gen_sample(model, in_tokens, max_new_tokens=100, context_len=1024)
print(token_ids_to_text(out_tokens, tokenizer))

## Notes

- The training script uses the existing instruction collator. Its Qwen padding and end-of-message behavior should be verified before relying on a full run.
- The script currently reports the last batch loss each epoch; it does not evaluate validation loss.
- Check the output directory before rerunning if you need to preserve a previous model.
